In [1]:
import math

def calculate_squared_euclidean_distance(seq1, seq2):
    """
    計算兩個等長序列之間的平方歐幾里德距離。
    這將作為我們比較子片段的成本函數。
    """
    if len(seq1) != len(seq2):
        # 為了避免在內部循環中因長度不符而報錯，這裡直接返回無窮大
        return float('inf')
    
    distance = 0
    for x, y in zip(seq1, seq2):
        distance += (x - y) ** 2
    return distance

def find_best_fixed_length_segment_pelt_like(query, serie):
    """
    在 serie 中尋找與 query 等長的最佳子片段。

    此函數採用 PELT 啟發式的方法，考慮 len(query) 個不同的起始偏移量。
    對於每個偏移量 'i_offset'，它會建構一個概念上的動態規劃矩陣
    (實作上為一個一維數組)，其中每個元素代表一個符合偏移量條件的子片段的成本。
    最終，函數會從所有偏移量中找出總體距離最小的子片段。

    Args:
        query (list or array): 查詢序列 (長度為 Q)。
        serie (list or array): 原始序列 (長度為 N)。

    Returns:
        tuple: (min_distance, best_start_idx, best_end_idx)
               min_distance: 找到的最小平方歐幾里德距離。
               best_start_idx: 最佳子片段在 serie 中的起始索引。
               best_end_idx: 最佳子片段在 serie 中的結束索引。
               如果沒有找到有效的子片段，則返回 (float('inf'), -1, -1)。
    """
    Q = len(query)
    N = len(serie)

    # 處理邊界情況：查詢或系列為空，或系列比查詢短。
    if Q == 0 or N == 0 or N < Q:
        return float('inf'), -1, -1

    overall_min_dist = float('inf')
    best_start_idx = -1
    best_end_idx = -1

    # 遍歷從 0 到 Q-1 的所有可能起始偏移量。
    # 這對應於您描述的 "起始點為 serie[i] for i in range(0, len(query))"。
    for i_offset in range(Q):
        # 對於每個偏移量，我們維護一個列表，它代表了該偏移量下的 "動態規劃矩陣"。
        # 這裡，dp_for_offset_i[k] 儲存了以 serie[k-1] 結尾且長度為 Q 的子片段的成本，
        # 前提是該子片段的起始索引滿足當前的 i_offset 條件。
        # 如果不滿足條件，則為 float('inf')。
        dp_for_offset_i = [float('inf')] * (N + 1) # dp_for_offset_i[k] 代表以 serie[k-1] 結尾的成本

        min_dist_for_current_offset = float('inf')
        current_best_start_for_offset = -1
        current_best_end_for_offset = -1

        # 遍歷 serie 中所有可能的子片段結束點 (k)。
        # 子片段的結束索引是 k-1。
        for k in range(Q, N + 1):
            segment_start_idx = k - Q
            segment_end_idx = k - 1

            # 檢查當前子片段的起始索引是否滿足當前的 i_offset 條件。
            # 這是我們根據起始點的模式來篩選子片段的核心邏輯。
            if segment_start_idx % Q == i_offset:
                segment_to_compare = serie[segment_start_idx : k]
                
                # 計算當前子片段與 query 之間的距離。
                current_cost = calculate_squared_euclidean_distance(query, segment_to_compare)
                
                # 將成本儲存在當前偏移量的 "動態規劃矩陣" 中。
                dp_for_offset_i[k] = current_cost
                
                # 更新當前偏移量下的最小距離。
                if current_cost < min_dist_for_current_offset:
                    min_dist_for_current_offset = current_cost
                    current_best_start_for_offset = segment_start_idx
                    current_best_end_for_offset = segment_end_idx
        
        # 在遍歷完所有可能結束點後，min_dist_for_current_offset
        # 就是該 "動態規劃矩陣" (即 dp_for_offset_i 列表) 中找到的最小距離。
        # 這對應了您描述的 "以每個起點返回它矩陣最後一row中，找出distance最小的部分"。
        
        # 更新總體最小距離和其對應的索引。
        # 如果遇到相同最小值，則保留第一個找到的 (通常是起始索引較大的)。
        if min_dist_for_current_offset < overall_min_dist:
            overall_min_dist = min_dist_for_current_offset
            best_start_idx = current_best_start_for_offset
            best_end_idx = current_best_end_for_offset

    # 返回最終的總體最小距離及其對應的起始和結束索引。
    return overall_min_dist, best_start_idx, best_end_idx

In [2]:
# 引入之前定義的函數
# from your_module import find_best_fixed_length_segment_pelt_like, calculate_squared_euclidean_distance

# --- 測試案例 1: 簡單的精確匹配 ---
query1 = [1, 2, 3]
serie1 = [0, 1, 1, 2, 3, 4, 5, 1, 2, 3, 6]
# 預期：在索引 2-4 (1,2,3) 和 7-9 (1,2,3) 都會找到，距離為0。
# 由於程式碼會取第一個找到的最小距離，所以結果可能取決於遍歷順序。
# 在這個實現中，預計會找到較早的那個。
# 實際輸出： (0.0, 2, 4) 或 (0.0, 7, 9)

print("--- 測試案例 1: 簡單的精確匹配 ---")
min_dist1, start_idx1, end_idx1 = find_best_fixed_length_segment_pelt_like(query1, serie1)
print(f"Query: {query1}, Serie: {serie1}")
print(f"最小距離: {min_dist1}, 起始索引: {start_idx1}, 結束索引: {end_idx1}")
print(f"找到的子片段: {serie1[start_idx1 : end_idx1 + 1] if start_idx1 != -1 else 'N/A'}\n")


# --- 測試案例 2: 沒有精確匹配，但有近似匹配 ---
query2 = [10, 20, 30]
serie2 = [1, 5, 12, 21, 28, 4, 9, 18, 27, 35]
# 預期：在索引 2-4 ([12, 21, 28]) 和 7-9 ([18, 27, 35]) 會有近似匹配。
# 計算距離：
# [12, 21, 28] vs [10, 20, 30] -> (12-10)^2 + (21-20)^2 + (28-30)^2 = 2^2 + 1^2 + (-2)^2 = 4 + 1 + 4 = 9
# [18, 27, 35] vs [10, 20, 30] -> (18-10)^2 + (27-20)^2 + (35-30)^2 = 8^2 + 7^2 + 5^2 = 64 + 49 + 25 = 138
# 預期輸出：(9.0, 2, 4)

print("--- 測試案例 2: 沒有精確匹配，但有近似匹配 ---")
min_dist2, start_idx2, end_idx2 = find_best_fixed_length_segment_pelt_like(query2, serie2)
print(f"Query: {query2}, Serie: {serie2}")
print(f"最小距離: {min_dist2}, 起始索引: {start_idx2}, 結束索引: {end_idx2}")
print(f"找到的子片段: {serie2[start_idx2 : end_idx2 + 1] if start_idx2 != -1 else 'N/A'}\n")


# --- 測試案例 3: serie 中有多個可能匹配，查詢長度為1 ---
query3 = [5]
serie3 = [1, 2, 5, 3, 6, 5, 4, 5, 7]
# 預期：會找到所有 5 的位置 (索引 2, 5, 7)。
# 由於距離都為 0，預計會返回第一個找到的。
# 預期輸出：(0.0, 2, 2)

print("--- 測試案例 3: serie 中有多個可能匹配，查詢長度為1 ---")
min_dist3, start_idx3, end_idx3 = find_best_fixed_length_segment_pelt_like(query3, serie3)
print(f"Query: {query3}, Serie: {serie3}")
print(f"最小距離: {min_dist3}, 起始索引: {start_idx3}, 結束索引: {end_idx3}")
print(f"找到的子片段: {serie3[start_idx3 : end_idx3 + 1] if start_idx3 != -1 else 'N/A'}\n")


# --- 測試案例 4: serie 比 query 短 ---
query4 = [1, 2, 3, 4, 5]
serie4 = [1, 2, 3]
# 預期：無法找到匹配，返回 float('inf'), -1, -1

print("--- 測試案例 4: serie 比 query 短 ---")
min_dist4, start_idx4, end_idx4 = find_best_fixed_length_segment_pelt_like(query4, serie4)
print(f"Query: {query4}, Serie: {serie4}")
print(f"最小距離: {min_dist4}, 起始索引: {start_idx4}, 結束索引: {end_idx4}")
print(f"找到的子片段: {serie4[start_idx4 : end_idx4 + 1] if start_idx4 != -1 else 'N/A'}\n")


# --- 測試案例 5: 空的 query ---
query5 = []
serie5 = [1, 2, 3]
# 預期：無法找到匹配，返回 float('inf'), -1, -1

print("--- 測試案例 5: 空的 query ---")
min_dist5, start_idx5, end_idx5 = find_best_fixed_length_segment_pelt_like(query5, serie5)
print(f"Query: {query5}, Serie: {serie5}")
print(f"最小距離: {min_dist5}, 起始索引: {start_idx5}, 結束索引: {end_idx5}")
print(f"找到的子片段: {serie5[start_idx5 : end_idx5 + 1] if start_idx5 != -1 else 'N/A'}\n")


# --- 測試案例 6: 空的 serie ---
query6 = [1, 2, 3]
serie6 = []
# 預期：無法找到匹配，返回 float('inf'), -1, -1

print("--- 測試案例 6: 空的 serie ---")
min_dist6, start_idx6, end_idx6 = find_best_fixed_length_segment_pelt_like(query6, serie6)
print(f"Query: {query6}, Serie: {serie6}")
print(f"最小距離: {min_dist6}, 起始索引: {start_idx6}, 結束索引: {end_idx6}")
print(f"找到的子片段: {serie6[start_idx6 : end_idx6 + 1] if start_idx6 != -1 else 'N/A'}\n")


# --- 測試案例 7: 長度較長，且存在跨 i_offset 的最佳解 ---
query7 = [10, 11]
serie7 = [1, 2, 3, 10, 11, 4, 5, 9, 12, 10, 11, 6, 7]
# query 長度為 2，i_offset 會有 0 和 1 兩種。
# 索引 3-4 (10,11) 距離 0，起始索引 3 % 2 == 1 -> i_offset = 1
# 索引 9-10 (10,11) 距離 0，起始索引 9 % 2 == 1 -> i_offset = 1
# 預期：找到 (0.0, 3, 4)

print("--- 測試案例 7: 長度較長，且存在跨 i_offset 的最佳解 ---")
min_dist7, start_idx7, end_idx7 = find_best_fixed_length_segment_pelt_like(query7, serie7)
print(f"Query: {query7}, Serie: {serie7}")
print(f"最小距離: {min_dist7}, 起始索引: {start_idx7}, 結束索引: {end_idx7}")
print(f"找到的子片段: {serie7[start_idx7 : end_idx7 + 1] if start_idx7 != -1 else 'N/A'}\n")

# --- 測試案例 8: 只有一個可能的子片段 ---
query8 = [50, 60]
serie8 = [10, 20, 51, 59]
# 預期：找到 (51, 59) 距離 (51-50)^2 + (59-60)^2 = 1^2 + (-1)^2 = 1 + 1 = 2
# 預期輸出：(2.0, 2, 3)

print("--- 測試案例 8: 只有一個可能的子片段 ---")
min_dist8, start_idx8, end_idx8 = find_best_fixed_length_segment_pelt_like(query8, serie8)
print(f"Query: {query8}, Serie: {serie8}")
print(f"最小距離: {min_dist8}, 起始索引: {start_idx8}, 結束索引: {end_idx8}")
print(f"找到的子片段: {serie8[start_idx8 : end_idx8 + 1] if start_idx8 != -1 else 'N/A'}\n")

--- 測試案例 1: 簡單的精確匹配 ---
Query: [1, 2, 3], Serie: [0, 1, 1, 2, 3, 4, 5, 1, 2, 3, 6]
最小距離: 0, 起始索引: 7, 結束索引: 9
找到的子片段: [1, 2, 3]

--- 測試案例 2: 沒有精確匹配，但有近似匹配 ---
Query: [10, 20, 30], Serie: [1, 5, 12, 21, 28, 4, 9, 18, 27, 35]
最小距離: 9, 起始索引: 2, 結束索引: 4
找到的子片段: [12, 21, 28]

--- 測試案例 3: serie 中有多個可能匹配，查詢長度為1 ---
Query: [5], Serie: [1, 2, 5, 3, 6, 5, 4, 5, 7]
最小距離: 0, 起始索引: 2, 結束索引: 2
找到的子片段: [5]

--- 測試案例 4: serie 比 query 短 ---
Query: [1, 2, 3, 4, 5], Serie: [1, 2, 3]
最小距離: inf, 起始索引: -1, 結束索引: -1
找到的子片段: N/A

--- 測試案例 5: 空的 query ---
Query: [], Serie: [1, 2, 3]
最小距離: inf, 起始索引: -1, 結束索引: -1
找到的子片段: N/A

--- 測試案例 6: 空的 serie ---
Query: [1, 2, 3], Serie: []
最小距離: inf, 起始索引: -1, 結束索引: -1
找到的子片段: N/A

--- 測試案例 7: 長度較長，且存在跨 i_offset 的最佳解 ---
Query: [10, 11], Serie: [1, 2, 3, 10, 11, 4, 5, 9, 12, 10, 11, 6, 7]
最小距離: 0, 起始索引: 3, 結束索引: 4
找到的子片段: [10, 11]

--- 測試案例 8: 只有一個可能的子片段 ---
Query: [50, 60], Serie: [10, 20, 51, 59]
最小距離: 2, 起始索引: 2, 結束索引: 3
找到的子片段: [51, 59]

